# Parser

In [ ]:
import os
import time
import requests
from urllib.parse import quote
from typing import Any

GITHUB_TOKEN = ''
if not GITHUB_TOKEN:
    raise ValueError("Необходимо задать GITHUB_TOKEN в переменных окружения")

HEADERS = {
    "Authorization": f"token {GITHUB_TOKEN}",
    "Accept": "application/vnd.github.v3+json"
}
TARGET_LICENSES = {"mit", "cc0-1.0", None}

DATASET_DIR = "Raw_Dataset"
os.makedirs(DATASET_DIR, exist_ok=True)

SEARCH_QUERIES = [
    "topic:python language:python",
    "topic:python-projects language:python",
]
MAX_PER_PAGE = 100
MAX_PAGES = 400
REQUEST_DELAY = 0.1
MAX_FILES = 25000

def search_repositories(query, page=1) -> tuple[Any, Any]:
    url = "https://api.github.com/search/repositories"
    params = {
        "q": query,
        "per_page": MAX_PER_PAGE,
        "page": page
    }
    resp = requests.get(url, headers=HEADERS, params=params)
    resp.raise_for_status()
    data = resp.json()
    return data["items"], data["total_count"]

def get_default_branch(owner, repo) -> Any:
    url = f"https://api.github.com/repos/{owner}/{repo}"
    resp = requests.get(url, headers=HEADERS)
    resp.raise_for_status()
    return resp.json()["default_branch"]

def get_repo_tree(owner, repo, branch) -> Any:
    url = f"https://api.github.com/repos/{owner}/{repo}/git/trees/{branch}?recursive=1"
    resp = requests.get(url, headers=HEADERS)
    resp.raise_for_status()
    return resp.json()["tree"]

def get_file_content(owner, repo, file_path, branch) -> Any | str | None:
    path_encoded = quote(file_path)
    url = f"https://api.github.com/repos/{owner}/{repo}/contents/{path_encoded}?ref={branch}"
    resp = requests.get(url, headers=HEADERS)
    if resp.status_code == 403 and "rate limit" in resp.text.lower():
        print("Request limit exceeded! Waiting")
        time.sleep(60)
        return get_file_content(owner, repo, file_path, branch)
    if resp.status_code != 200:
        print(f"Unable to get file {file_path} (status {resp.status_code})")
        return None
    data = resp.json()
    if data["type"] != "file":
        return None
    download_url = data["download_url"]
    file_resp = requests.get(download_url, headers=HEADERS)
    if file_resp.status_code != 200:
        print(f"Unable to load file: {file_path}")
        return None
    return file_resp.text

def save_file(content, repo_full_name, file_path) -> None:
    safe_name = f"{repo_full_name.replace('/', '__')}__{file_path.replace('/', '_')}"
    if len(safe_name) > 250:
        safe_name = safe_name[:250] + ".py"
    filepath = os.path.join(DATASET_DIR, safe_name)
    with open(filepath, "w", encoding="utf-8") as f:
        f.write(content)
    print(f"Saved: {safe_name}")


def main() -> None:
    page = 1
    total_downloaded = 0
    processed_repos = 0

    for search_query in SEARCH_QUERIES:
        print('\n' * 3)
        print(f"Searching for: {search_query}")
        print('\n' * 3)

        while total_downloaded < MAX_FILES:
            print(f"Loading {page}...")
            try:
                items, total_count = search_repositories(search_query, page)
            except Exception as e:
                print(f"Error while searching: {e}")
                break

            if not items:
                print("No repos on the page")
                break

            print(f"Found repos on the page: {len(items)} (sum: {total_count})")

            for repo in items:
                repo_full_name = repo["full_name"]
                license_info = repo.get("license")
                license_key = license_info["key"] if license_info else None

                if license_key not in TARGET_LICENSES:
                    continue
                try:
                    branch = get_default_branch(*repo_full_name.split('/'))
                    time.sleep(REQUEST_DELAY)

                    tree = get_repo_tree(*repo_full_name.split('/'), branch)
                    time.sleep(REQUEST_DELAY)

                    py_files = [item for item in tree if item["type"] == "blob" and item["path"].endswith(".py")]
                    if not py_files:
                        print(" No .py files")
                        continue
                    print(f"Found .py files: {len(py_files)} if repo {repo_full_name}")

                    for file_item in py_files:
                        file_path = file_item["path"]
                        content = get_file_content(*repo_full_name.split('/'), file_path, branch)
                        if content is not None:
                            save_file(content, repo_full_name, file_path)
                            total_downloaded += 1
                        time.sleep(REQUEST_DELAY)

                except Exception as e:
                    print(f"Error while parsing repo: {e}")

                processed_repos += 1

            if len(items) < MAX_PER_PAGE:
                break 
            page += 1
            time.sleep(REQUEST_DELAY * 2) 

        print()
        print(f"Done! Parsed repos: {processed_repos}, files downloaded: {total_downloaded}")

main()